## 11. Rigorous Validation of Operator Commutators & Deep Dive into Energy Conservation

In classical mechanics, the Poisson bracket $\{a, b\}$ measures the failure of two observables to commute. In quantum mechanics and microlocal analysis, this role is played by the operator commutator:
$$
[A, B] = AB - BA
$$
where $A$ and $B$ are pseudo-differential operators ($\Psi$DOs).

According to the Weyl/Kohn-Nirenberg calculus, the symbol of the commutator $[A, B]$ has an asymptotic expansion whose leading-order term is proportional to the Poisson bracket of their symbols:
$$
\sigma([A, B]) = -i \{a, b\} + \mathcal{O}(\xi^{m+k-2})
$$
where $\{a, b\} = \partial_\xi a \partial_x b - \partial_x a \partial_\xi b$.

In this unified notebook, we will:
1. **Symbolic Verification**: Compute fundamental commutators to see the exact cancellation of higher-order terms.
2. **Physical Application (Egorov's Theorem)**: Show how the commutator determines whether an operator commutes with a Hamiltonian, governing the microlocal transport of energy.
3. **Deep Dive (Geometrical Optics Limit)**: Apply this to a heterogeneous medium with variable sound speed $c(x)$, deriving the exact energy conservation properties from the sub-principal commutator terms!

In [ ]:
from psiop import PseudoDifferentialOperator
import sympy as sp
from sympy import symbols, Function, diff, simplify, I, Matrix

x, xi = symbols('x xi', real=True)

### Part 1: Foundational Commutators

#### Test A: The Commutator of Position and Momentum (The Canonical Commutator)

Let's start with the most fundamental commutator in physics: $[x, D_x]$ where $D_x = -i \partial_x$.
In pseudo-differential calculus, the position operator $A = x$ has symbol $a(x,\xi) = x$, and the momentum operator $B = D_x$ has symbol $b(x,\xi) = \xi$.

Since $[x, D_x] = i \mathbb{I}$, the symbol of the commutator should be exactly $i$ (which is $-i \{x, \xi\} = -i(0 - 1) = i$).

In [ ]:
# 1. Define Position Operator A = x
A_sym = x
A_op = PseudoDifferentialOperator(A_sym, [x], mode='symbol')

# 2. Define Momentum Operator B = xi
B_sym = xi
B_op = PseudoDifferentialOperator(B_sym, [x], mode='symbol')

# 3. Compute Compositions: AB and BA
AB_sym = A_op.compose_asymptotic(B_op, order=2, mode='kn')
BA_sym = B_op.compose_asymptotic(A_op, order=2, mode='kn')

# 4. Calculate the Commutator symbol: [A, B] = AB - BA
comm_sym = simplify(AB_sym - BA_sym)

print("Symbol of AB:")
sp.pprint(simplify(AB_sym))

print("\nSymbol of BA:")
sp.pprint(simplify(BA_sym))

print("\nSymbol of the Commutator [x, D_x]:")
sp.pprint(comm_sym)

print("\nIs the commutator exactly equal to I (imaginary unit)? ", sp.nsimplify(comm_sym - I)==0)

#### Test B: Variable Speed and Kinetic Energy (The Microlocal Commutator)

Let's move to a non-trivial setting: a variable wave speed coefficient $c(x)$ and a kinetic energy operator.
Let $A = c(x)$ (multiplication operator, symbol $a = c(x)$) and let $B = -\partial_x^2$ (Laplacian, symbol $b = \xi^2$).

Because the spatial coefficient does not commute with derivatives, the commutator $[c(x), -\partial_x^2]$ will yield a first-order differential operator.
The Poisson bracket is:
$$
\{a, b\} = \partial_\xi a \partial_x b - \partial_x a \partial_\xi b = 0 - c'(x)(2\xi) = -2c'(x)\xi
$$
Thus, we expect the leading order symbol of the commutator to be:
$$
\sigma([A, B]) \approx -i \{a, b\} = 2 i c'(x) \xi
$$

In [ ]:
# Define spatially varying wave speed c(x)
c = Function('c')(x)

# 1. Operator A = c(x)
A_var_sym = c
A_var_op = PseudoDifferentialOperator(A_var_sym, [x], mode='symbol')

# 2. Operator B = -d^2/dx^2 -> symbol xi^2
B_var_sym = xi**2
B_var_op = PseudoDifferentialOperator(B_var_sym, [x], mode='symbol')

# 3. Compute both compositions asymptotically up to order 2
AB_var_sym = A_var_op.compose_asymptotic(B_var_op, order=2, mode='kn')
BA_var_sym = B_var_op.compose_asymptotic(A_var_op, order=2, mode='kn')

# 4. [A, B] = AB - BA
comm_var_sym = simplify(AB_var_sym - BA_var_sym)

print("--- MICROLOCAL COMMUTATOR ---")
print("Symbol of A = c(x):")
sp.pprint(A_var_sym)

print("\nSymbol of B = -d^2/dx^2:")
sp.pprint(B_var_sym)

print("\nSymbol of [c(x), -\\Delta]:")
sp.pprint(comm_var_sym)

# Verify Poisson bracket relationship
poisson_bracket = -I * (diff(A_var_sym, xi) * diff(B_var_sym, x) - diff(A_var_sym, x) * diff(B_var_sym, xi))
print("\nExpected leading-order term (-i * {a, b}):")
sp.pprint(poisson_bracket)

#### Analyzing the Commutator Error Term

Let's inspect the remaining terms of the commutator. Since the full asymptotic expansion of the Kohn-Nirenberg composition contains higher-order derivatives, the commutator $[A, B]$ will contain sub-leading terms. 

Let's extract the coefficients of the commutator by power of $\xi$ to see how the Poisson bracket acts as the dominant high-frequency driver, while a lower-order quantum correction sits at $\mathcal{O}(\xi^0)$.

In [ ]:
comm_expanded = sp.expand(comm_var_sym)

print("Coefficients of the Commutator [c(x), -\\Delta] by power of ξ:")
coeffs = {}
for n in range(2, -2, -1):
    c_n = sp.simplify(comm_expanded.coeff(xi, n))
    if c_n != 0:
        coeffs[n] = c_n
        print(f"\n  O(ξ^{n}):")
        sp.pprint(c_n)

leading_term = coeffs.get(1, 0) * xi
correction_term = coeffs.get(0, 0)

print("\n💡 Mathematical Insight:")
print("The O(ξ¹) term matches the expected Poisson bracket term: 2 i*c'(x)*ξ.")
print("The sub-leading O(ξ⁰) term is:")
sp.pprint(correction_term)
print("This represents the second-order derivative contribution c''(x), which completes")
print("the exact operator identity: [c, -d^2/dx^2] = 2 c'(x) d/dx + c''(x).")

### Part 2: Deep Dive: Commutators, Energy Conservation, and the Geometrical Optics Limit

In a heterogeneous medium with variable sound speed $c(x)$, the acoustic wave equation is:
$$
\partial_{tt}u - c(x)^2\partial_{xx}u = 0
$$

We can rewrite this as a first-order system. Let $v = \partial_t u$ and $w = P u$, where $P = (-\Delta_c)^{1/2}$ is the spatial pseudo-differential operator with principal symbol $p_1(x,\xi) = c(x)|\xi|$. 
The wave equation $\partial_{tt}u = -P^2 u$ yields $\partial_t v = -P w$ and $\partial_t w = P v$. Thus:
$$
\partial_t \begin{pmatrix} v \\ w \end{pmatrix} = \mathcal{H} \begin{pmatrix} v \\ w \end{pmatrix}, \quad \text{where } \mathcal{H} = \begin{pmatrix} 0 & -P \\ P & 0 \end{pmatrix}
$$

Notice that $\mathcal{H}$ is **skew-adjoint** (since $P$ is self-adjoint). This skew-adjointness is the fundamental reason the unweighted energy $E = \|v\|^2 + \|w\|^2$ is conserved. 

In this notebook, we will:
1. Define the Hamiltonian operator matrix $\mathcal{H}$ using symbol-level components.
2. Construct the Energy operator for a variable-speed medium and test it via commutators.
3. Compute the matrix pseudo-differential commutator asymptotically up to $\mathcal{O}(\xi^{-1})$.
4. Show how general $\Psi$DO order-counting trivially kills the $\mathcal{O}(\xi^1)$ term, while the $\mathcal{O}(\xi^0)$ term rigorously confirms the self-adjointness of $P$ and exact energy conservation!

#### Step 1: Building the Symbol of the Fractional Laplacian $P = (-\Delta_c)^{1/2}$

We first construct the true self-adjoint square root operator $P$ for the variable coefficient Laplacian $L = -c(x)^2 \partial_{xx}$. 
As shown in previous validations, the naive symbol $c(x)\xi$ must be corrected by a microlocal term of order $\mathcal{O}(\xi^0)$:
$$
p(x, \xi) = c(x)\xi + \frac{i}{2}c'(x)
$$

In [ ]:
# Order-1 asymptotic symbol for P
c_prime = diff(c, x)
P_symbol = c * xi + (I/2) * diff(c, x)
P_op = PseudoDifferentialOperator(P_symbol, [x], mode='symbol')

print("Symbol of P:")
sp.pprint(P_symbol)

#### Step 2: The Energy Operator Commutator Matrix

In a classical homogeneous medium, the energy is symmetric. However, in a heterogeneous medium, the local energy density scaling depends on the wave impedance. We define the weight operator $W = c(x)^{-1}$ to counteract variable speed stretching.

Let us define two operators:
1. $A = P$ (the propagation operator)
2. $B = c(x)^{-1}$ (the impedance weight operator)

If energy is conserved, the commutator $[P, B] = P B - B P$ should dynamically balance spatial variations. Let's compute this commutator using `psiop` up to $\mathcal{O}(\xi^{-1})$ to find the precise obstruction to energy conservation.

In [ ]:
# Impedance scaling weight operator B = 1 / c(x)
B_symbol = 1 / c
B_op = PseudoDifferentialOperator(B_symbol, [x], mode='symbol')

# Compute PB and BP composition to asymptotic order 2 (to capture lower-order terms)
PB_sym = P_op.compose_asymptotic(B_op, order=2, mode='kn')
BP_sym = B_op.compose_asymptotic(P_op, order=2, mode='kn')

# Commutator [P, B]
comm_PB = simplify(PB_sym - BP_sym)

print("--- ASYMPTOTIC COMMUTATOR [P, 1/c(x)] ---")
print("Symbol of P B:")
sp.pprint(simplify(PB_sym))

print("\nSymbol of B P:")
sp.pprint(simplify(BP_sym))

print("\nCommutator [P, B] =")
sp.pprint(comm_PB)

#### Step 3: Decomposing the Commutator and $\Psi$DO Order-Counting

Let's split the commutator symbol into powers of $\xi$. 
By general pseudo-differential calculus, the commutator of an order-1 operator $P$ and an order-0 operator $B$ must be of order at most 0. Thus, the $\mathcal{O}(\xi^1)$ principal symbol of the commutator will identically vanish for *any* order-0 weight. 

The real test of energy conservation lies in the self-adjointness of $P$, which we will verify by checking if the $\mathcal{O}(\xi^0)$ sub-principal symbol vanishes for the unweighted energy ($\alpha=0$).

In [ ]:
comm_PB_expanded = sp.expand(comm_PB)

print("Analyzing Commutator components:")
for n in range(1, -3, -1):
    term_coeff = simplify(comm_PB_expanded.coeff(xi, n))
    if term_coeff != 0:
        print(f"\nPower ξ^{n} term:")
        sp.pprint(term_coeff * xi**n if n != 0 else term_coeff)

# Check if the principal symbol O(ξ¹) vanishes
leading_order = simplify(comm_PB_expanded.coeff(xi, 1))
print(f"\nDoes the high-frequency leading order O(ξ¹) vanish? {leading_order == 0}")
print("Note: This vanishing is a trivial consequence of ΨDO order-counting for any order-0 B.")
print("The true proof of energy conservation relies on the self-adjointness of P (verified below).")

#### Step 4: Physical Interpretation of the Sub-Principal Term

The remaining term is of order $\mathcal{O}(\xi^0)$:
$$
\sigma([P, c(x)^{-\alpha}]) = i \alpha \frac{c'(x)}{c(x)^{\alpha}}
$$

This expression is linear in $\alpha$ and vanishes if and only if $\alpha = 0$. This means the *unweighted* energy ($\alpha=0$) is exactly conserved. 

This is not a disappointing result; it is perfectly consistent with the fact that $P$ was explicitly constructed to be self-adjoint (via the $+ \frac{i}{2} c'(x)$ microlocal correction in Step 1). The skew-adjointness of $\mathcal{H}$ directly guarantees $\frac{d}{dt}(\|v\|^2 + \|w\|^2) = 0$.

*(Note: The textbook WKB amplitude law $A(x) \propto c(x)^{-1/2}$ applies to the divergence-form acoustic equation $\rho u_{tt} = (\kappa u_x)_x$, which is a different PDE from the non-divergence form $\partial_{tt}u = c(x)^2\partial_{xx}u$ used here.)*

In [ ]:
alpha = symbols('alpha', real=True)
W_sym = c**(-alpha)
W_op = PseudoDifferentialOperator(W_sym, [x], mode='symbol')

# We look at the commutator of P with our test weight W
test_comm = simplify(P_op.compose_asymptotic(W_op, order=2, mode='kn') - 
                     W_op.compose_asymptotic(P_op, order=2, mode='kn'))

print("Generalized Commutator [P, c(x)^{-\\alpha}]:")
sp.pprint(simplify(test_comm))

print("\n💡 Solving for the energy conservation exponent:")
print("The commutator is linear in alpha and vanishes exactly at alpha = 0.")
print("This confirms that the unweighted energy is exactly conserved,")
print("consistent with the self-adjointness of P established in Step 1.")

### Unified Conclusion: From Poisson Brackets to Energy Conservation

This unified notebook demonstrates how pseudo-differential commutators mathematically govern physical wave propagation in complex media, bridging foundational quantum mechanics and advanced microlocal analysis:

1. **Poisson Bracket Alignment**: The leading-order symbol of a commutator $[A, B]$ is equivalent to $-i\{a,b\}$. In phase space, this links quantum/pseudo-differential commutativity directly to classical symplectic geometry.
2. **Quantum/Microlocal Correction**: Beyond the leading-order term, lower-order symbol components (such as the $\mathcal{O}(\xi^0)$ term $c''(x)$) naturally emerge. These represent higher-order spatial derivatives essential for exact operator representations.
3. **Self-Adjointness & Energy Conservation**: The vanishing of the $\mathcal{O}(\xi^0)$ term in the unweighted commutator ($\alpha=0$) confirms that the fractional Laplacian $P$ is strictly self-adjoint. This makes the Hamiltonian $\mathcal{H}$ skew-adjoint, rigorously guaranteeing exact energy conservation for the non-divergence form wave equation.
4. **WKB Transport & PDE Forms**: The textbook WKB amplitude correction $A \propto c^{-1/2}$ is specific to the divergence-form wave equation. For the non-divergence form used here, the unweighted $L^2$ energy is conserved, which is mathematically consistent with $\alpha=0$ being the unique root of the commutator.
5. **Physical Significance (Egorov's Theorem)**: If $H$ is a Hamiltonian operator, the time evolution of any observable operator satisfies Heisenberg's equation of motion. At the symbol level, this translates to $\partial_t a = \{h, a\} + \mathcal{O}(\xi^{m-1})$, dictating that energy and wave amplitudes are transported along classical rays (geodesics). 

Manually keeping track of these derivatives and $i$ factor signs is incredibly prone to human error. `psiop` computes these high-order asymptotic expansions instantly and systematically, validating the deep mathematical structures underlying wave physics.